In [1]:
import torch
import torch.nn as nn
import math
from torch.utils.data import Dataset, DataLoader
from collections import Counter

# ---- 数据准备（复用上一节） ----
texts = [
    "这部电影真的很好看", "非常精彩的一部作品", "太无聊了浪费时间",
    "剧情很差不推荐", "演员演技很棒值得一看", "烂片一部浪费钱",
    "感人至深的好电影", "剧情拖沓毫无亮点",
]
labels = [1, 1, 0, 0, 1, 0, 1, 0]


def tokenize(text):
    return list(text)


counter = Counter()
for t in texts:
    counter.update(tokenize(t))
vocab = {w: i + 2 for i, (w, _) in enumerate(counter.most_common())}
vocab_size = len(vocab) + 2


def encode(text, max_len=15):
    tokens = [vocab.get(c, 1) for c in tokenize(text)]
    tokens = tokens[:max_len] + [0] * max(0, max_len - len(tokens))
    return tokens


class TextDataset(Dataset):
    def __init__(self, texts, labels, max_len=15):
        self.X = [encode(t, max_len) for t in texts]
        self.y = labels

    def __len__(self): return len(self.y)

    def __getitem__(self, i):
        return torch.tensor(self.X[i]), torch.tensor(self.y[i])


dataset = TextDataset(texts, labels)
loader = DataLoader(dataset, batch_size=4, shuffle=True)


# ---- 位置编码 ----
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]


# ---- Transformer 分类模型 ----
class TransformerClassifier(nn.Module):
    def __init__(self, vocab_size, d_model=64, nhead=4, num_layers=2, num_classes=2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos = PositionalEncoding(d_model)
        layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead,
                                           dim_feedforward=128, batch_first=True)
        self.encoder = nn.TransformerEncoder(layer, num_layers=num_layers)
        self.fc = nn.Linear(d_model, num_classes)

    def forward(self, x):
        x = self.embedding(x)
        x = self.pos(x)
        x = self.encoder(x)
        return self.fc(x[:, 0, :])  # 取第一个位置做分类


# ---- 训练 ----
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = TransformerClassifier(vocab_size).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

epochs = 30
for epoch in range(epochs):
    model.train()
    total_loss = 0
    for batch_x, batch_y in loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        logits = model(batch_x)
        loss = criterion(logits, batch_y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch + 1}/{epochs}, Loss: {total_loss / len(loader):.4f}")

Epoch 10/30, Loss: 0.0293
Epoch 20/30, Loss: 0.0040
Epoch 30/30, Loss: 0.0023


In [2]:
@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    correct, total = 0, 0

    for batch_x, batch_y in loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        logits = model(batch_x)
        preds = logits.argmax(dim=1)
        correct += (preds == batch_y).sum().item()
        total += batch_y.size(0)

    return correct / total

In [3]:
acc = evaluate(model, loader, device)

In [4]:
print(f"准确率: {acc:.2%}")

准确率: 100.00%


In [5]:
def predict(text, model, vocab, device, max_len=15):
    model.eval()

    tokens = [vocab.get(c, 1) for c in text]
    tokens = tokens[:max_len] + [0] * max(0, max_len - len(tokens))

    x = torch.tensor([tokens]).to(device)
    with torch.no_grad():
        logits = model(x)
        prob = torch.softmax(logits, dim=1)

    return "正面" if logits.argmax(dim=1).item() == 1 else "负面", prob.max().item()

In [6]:
test_texts = ["这部片子太棒了", "真心不怎么样", "演技在线剧情紧凑"]

In [7]:
for t in test_texts:
    label, conf = predict(t, model, vocab, device)
    print(f"「{t}」 → {label} (置信度: {conf:.2%})")

「这部片子太棒了」 → 正面 (置信度: 99.59%)
「真心不怎么样」 → 正面 (置信度: 69.55%)
「演技在线剧情紧凑」 → 正面 (置信度: 99.74%)
